<a href="https://colab.research.google.com/github/anish165/this-is-my-new-repo/blob/main/QuantYog_Essentials_Binance_data_download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#quantyog.com
from google.colab import drive
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import math
import os.path
import pandas as pd
from binance.client import Client
from datetime import datetime, timezone
from dateutil import parser
import requests
from typing import *
import time

import warnings
warnings.filterwarnings("ignore")

%matplotlib inline

#%reload_ext autoreload
#%autoreload 2

In [ ]:
#QuantYog™
# Mount Google Drive
drive.mount("/content/drive/", force_remount = True)
%cd /content/drive/My Drive/QuantYog Essentials/Data

Mounted at /content/drive/
/content/drive/My Drive/QuantYog Essentials/Data


####**Important note** Please run the below code on your local system as opposed to google colab notebook

####**User Inputs**

In [ ]:
#linkedin.com/company/quantyog
binance_api_key = ''    #Enter your own API-key here - create an account with Binance and enter the api key from your account here
binance_api_secret = '' #Enter your own API-secret here - create an account with Binance and enter the api secret key from your account here
interval = ["1m"] #Enter the frequency of data you want to download. 1m = minutely data
folder_path_to_save = 'your_folder//' #Enter the folder path where you want to save the data
file_name_to_save = '%s-%s-data.parquet' #This format saves data as parquet (recommended)
binance_symbols = ['BTCUSDT'] #Enter all the symbols for which data has to be downloaded
binsizes = {"1m": 1, "1h": 60, "1d": 1440} # Binsize is the number of minutes in that interval
batch_size = 750 #In one fetch from the API call, these many data would be downloaded.
binance_client = Client(api_key=binance_api_key, api_secret=binance_api_secret) #Binance client is created here

####**Useful functions**

In [ ]:
#instagram.com/quantyog
"""
Please do not edit the below classes and functions. They are required to create an instance of the Binance Client
and then interact with the Binance hub to pull data

"""
class BinanceClient:
    def __init__(self, futures=False):
        self.exchange = "BINANCE"
        self.futures = futures

        if self.futures:
            self._base_url = "https://fapi.binance.com"
        else:
            self._base_url = "https://api.binance.com"

        self.symbols = self._get_symbols()

    def _make_request(self, endpoint: str, query_parameters: Dict):
        try:
            response = requests.get(self._base_url + endpoint, params=query_parameters)
        except Exception as e:
            print("Connection error while making request to %s: %s", endpoint, e)
            return None

        if response.status_code == 200:
            return response.json()
        else:
            print("Error while making request to %s: %s (status code = %s)",
                         endpoint, response.json(), response.status_code)
            return None

    def _get_symbols(self) -> List[str]:

        params = dict()

        endpoint = "/fapi/v1/exchangeInfo" if self.futures else "/api/v3/exchangeInfo"
        data = self._make_request(endpoint, params)

        symbols = [x["symbol"] for x in data["symbols"]]

        return symbols

    def get_historical_data(self, symbol: str, interval: Optional[str] = "1m", start_time: Optional[int] = None, end_time: Optional[int] = None, limit: Optional[int] = 1500):

        params = dict()

        params["symbol"] = symbol
        params["interval"] = interval
        params["limit"] = limit

        if start_time is not None:
            params["startTime"] = start_time
        if end_time is not None:
            params["endTime"] = end_time

        endpoint = "/fapi/v1/klines" if self.futures else "/api/v3/klines"
        raw_candles = self._make_request(endpoint, params)

        candles = []

        if raw_candles is not None:
            for c in raw_candles:
                candles.append((float(c[0]), float(c[1]), float(c[2]), float(c[3]), float(c[4]), float(c[5]),))
            return candles
        else:
            return None

def minutes_of_new_data_for_perps(symbol, interval, data, source):
    if len(data) > 0:  old = parser.parse(data.index[-1].strftime('%Y-%m-%d %H:%M:%S'))
    elif source == "binance": old = datetime.strptime('1 Jan 2017 00:00:00', '%d %b %Y %H:%M:%S')
    if source == "binance": new = pd.to_datetime(binance_client.futures_klines(symbol=symbol, interval=interval)[-1][0], unit='ms')
    return old, new

def ms_to_dt_utc(ms: int) -> datetime:
    return datetime.utcfromtimestamp(ms / 1000)

def ms_to_dt_local(ms: int) -> datetime:
    return datetime.fromtimestamp(ms / 1000)

def GetDataFrame(data):
    df = pd.DataFrame(data, columns=['Timestamp', "Open", "High", "Low", "Close", "Volume"])
    df["Timestamp"] = df["Timestamp"].apply(lambda x: ms_to_dt_utc(x))
    df['Date'] = df["Timestamp"].dt.strftime("%d/%m/%Y")
    df['Time'] = df["Timestamp"].dt.strftime("%H:%M:%S")
    column_names = ["Date", "Time", "Open", "High", "Low", "Close", "Volume"]
    df = df.set_index('Timestamp')
    df = df.reindex(columns=column_names)

    return df

def GetHistoricalData(client, symbol, interval, start_time, end_time, limit=1500):
    collection = []

    while start_time < end_time:
        data = client.get_historical_data(symbol, interval = interval, start_time=start_time, end_time=end_time, limit=limit)
        print(client.exchange + " " + symbol + " : Collected " + str(len(data)) + " initial data from "+ str(ms_to_dt_utc(data[0][0])) +" to " + str(ms_to_dt_utc(data[-1][0])))
        start_time = int(data[-1][0] + 1000)
        collection +=data
        time.sleep(1.1)

    return collection


####**Data Download**

In [ ]:
#linkedin.com/company/quantyog
for symbol in binance_symbols:
    client = BinanceClient(futures=True) #create the instance to download futures data
    saving_symbol = symbol + 'PERP'
    for j in interval:
        filename = folder_path_to_save + file_name_to_save % (saving_symbol, j)
        if os.path.isfile(filename):
            data_df = pd.read_parquet(filename) #this will read the amount of data already saved at your local disc, so that data is only downloaded from the last saved data point till present
        else: data_df = pd.DataFrame()

        oldest_point, newest_point = minutes_of_new_data_for_perps(symbol, j, data_df, source = "binance")

        delta_min = (newest_point - oldest_point).total_seconds()/60
        if delta_min !=0 :

            available_data = math.ceil(delta_min/binsizes[j])
            if oldest_point == datetime.strptime('1 Jan 2017', '%d %b %Y'): print('Downloading all available %s data for %s. Be patient..!' % (j, symbol))
            else: print('Downloading %d minutes of new data available for %s, i.e. %d instances of %s data.' % (delta_min, symbol, available_data, j))

            fromDate = int(oldest_point.replace(tzinfo=timezone.utc).timestamp() * 1000) #.strftime("%d %b %Y %H:%M:%S") #this calculates the start point for data to be downloaded
            toDate = int(newest_point.replace(tzinfo=timezone.utc).timestamp() * 1000) #.strftime("%d %b %Y %H:%M:%S") #this calculates the end point for data to be downloaded

            data = GetHistoricalData(client, symbol, j, fromDate, toDate) #data is being fetched
            data = GetDataFrame(data) # data is converted into a usable dataframe object
            if len(data_df) > 0:
                temp_df = pd.DataFrame(data)
                data_df = data_df.append(temp_df)
            else: data_df = data

            parquetfilename = folder_path_to_save + file_name_to_save % (saving_symbol, j)
            print(parquetfilename)
            data_df['Open'] = pd.to_numeric(data_df['Open']) # data is converted to numeric format
            data_df['High'] = pd.to_numeric(data_df['High'])
            data_df['Low'] = pd.to_numeric(data_df['Low'])
            data_df['Close'] = pd.to_numeric(data_df['Close'])
            data_df['Volume'] = pd.to_numeric(data_df['Volume'])


            data_df.to_parquet(parquetfilename) # data is saved to disc
            print('All saved..!')